In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
base_path = "abfss://sellers@adlsstdout001tgt.dfs.core.windows.net/Sellers/"


In [0]:
import builtins
from pyspark.sql.functions import current_date, lit

# ---- Year ----
folders = dbutils.fs.ls(base_path)
latest_year = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Month ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/")
latest_month = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Day ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/{latest_month}/")
latest_day = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Final Path ----
latest_path = f"{base_path}{latest_year}/{latest_month}/{latest_day}/"

print(f"Reading from: {latest_path}")

# ---- File Date ----
file_date = f"{latest_year}-{latest_month}-{latest_day}"

# ---- Read parquet ----
raw_df = (
    spark.read
    .parquet(latest_path)
    .withColumn("ingest_date", current_date())
    .withColumn("file_date", lit(file_date))
)

display(raw_df.limit(10))

# Data Cleaning

In [0]:
from pyspark.sql.window import Window
# ==========================================
# 2. STANDARDIZE COLUMN NAMES
# ==========================================

df = (
    raw_df
    .withColumnRenamed("seller_id", "seller_id")
    .withColumnRenamed("seller_zip_code_prefix", "seller_zip_code_prefix")
    .withColumnRenamed("seller_city", "seller_city")
    .withColumnRenamed("seller_state", "seller_state")
)

# ==========================================
# 3. REMOVE DUPLICATES
# BUSINESS KEY = seller_id
# KEEP LATEST RECORD
# ==========================================

window_spec = Window.partitionBy("seller_id") \
                    .orderBy(F.col("file_date").desc())

df = (
    df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# ==========================================
# 4. TRIM STRING COLUMNS
# ==========================================

string_cols = [
    "seller_id",
    "seller_city",
    "seller_state"
]

for col_name in string_cols:
    df = df.withColumn(col_name, F.trim(F.col(col_name)))

# ==========================================
# 5. HANDLE NULL VALUES
# ==========================================

df = (
    df
    .filter(F.col("seller_id").isNotNull())
    .filter(F.col("seller_zip_code_prefix").isNotNull())
)

# ==========================================
# 6. STANDARDIZE TEXT FORMATTING
# ==========================================

df = (
    df
    .withColumn(
        "seller_city",
        F.initcap(F.lower(F.col("seller_city")))
    )
    .withColumn(
        "seller_state",
        F.upper(F.col("seller_state"))
    )
)

# ==========================================
# 7. CLEAN ZIP CODE
# INDUSTRY STANDARD:
# - Keep only digits
# - Left pad to 5 chars
# ==========================================

df = (
    df
    .withColumn(
        "seller_zip_code_prefix",
        F.regexp_replace(
            F.col("seller_zip_code_prefix").cast("string"),
            "[^0-9]",
            ""
        )
    )
    .withColumn(
        "seller_zip_code_prefix",
        F.lpad(F.col("seller_zip_code_prefix"), 5, "0")
    )
)

# ==========================================
# 8. DATA VALIDATION RULES
# ==========================================

valid_states = [
    "AC","AL","AP","AM","BA","CE","DF","ES","GO",
    "MA","MT","MS","MG","PA","PB","PR","PE","PI",
    "RJ","RN","RS","RO","RR","SC","SP","SE","TO"
]

df = (
    df
    .filter(F.length("seller_id") == 32)
    .filter(F.length("seller_zip_code_prefix") == 5)
    .filter(F.col("seller_state").isin(valid_states))
)

# ==========================================
# 9. DATE STANDARDIZATION
# ==========================================

df = (
    df
    .withColumn(
        "ingest_date",
        F.to_date(F.col("ingest_date"))
    )
    .withColumn(
        "file_date",
        F.to_date(F.col("file_date"))
    )
)

# ==========================================
# 10. ADD AUDIT COLUMNS
# ==========================================

df_cleaned = (
    df
    .withColumn("created_timestamp", F.current_timestamp())
    .withColumn("updated_timestamp", F.current_timestamp())
)

# ==========================================
# 11. FINAL COLUMN ORDER
# ==========================================

df_cleaned = df_cleaned.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "ingest_date",
    "file_date",
    "created_timestamp",
    "updated_timestamp"
)



In [0]:
table_name = "ecom.silver.dim_silver_sellers"

In [0]:
from pyspark.sql.functions import *

new_df = (
    df_cleaned
    .withColumn("seller_dim_id", expr("uuid()"))
    .withColumn("is_active", lit(1))
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast("date"))
)

In [0]:
if spark.catalog.tableExists(table_name):

    # ==========================================
    # CREATE TEMP VIEW
    # ==========================================

    new_df.createOrReplaceTempView("new_df")

    # ==========================================
    # STEP 1 : EXPIRE OLD RECORDS
    # ==========================================

    spark.sql("""
    MERGE INTO ecom.silver.dim_sellers AS t
    USING new_df AS s

    ON
        t.seller_id = s.seller_id
        AND t.is_active = 1

    WHEN MATCHED AND (
        t.seller_zip_code_prefix <> s.seller_zip_code_prefix OR
        t.seller_city <> s.seller_city OR
        t.seller_state <> s.seller_state
    )

    THEN UPDATE SET
        t.is_active = 0,
        t.end_date = current_date(),
        t.updated_timestamp = current_timestamp()

    """)

    # ==========================================
    # STEP 2 : INSERT NEW + CHANGED RECORDS
    # ==========================================

    spark.sql("""
    MERGE INTO ecom.silver.dim_sellers AS t
    USING new_df AS s

    ON
        t.seller_id = s.seller_id
        AND t.is_active = 1

    WHEN NOT MATCHED

    THEN INSERT (

        seller_dim_id,
        seller_id,
        seller_zip_code_prefix,
        seller_city,
        seller_state,
        ingest_date,
        file_date,
        created_timestamp,
        updated_timestamp,
        is_active,
        start_date,
        end_date

    )

    VALUES (

        s.seller_dim_id,
        s.seller_id,
        s.seller_zip_code_prefix,
        s.seller_city,
        s.seller_state,
        s.ingest_date,
        s.file_date,
        current_timestamp(),
        current_timestamp(),
        1,
        current_date(),
        NULL

    )

    """)


else:
    (
        new_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option(
                    "path",
                    "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/Sellers"
                )
        .saveAsTable(table_name)
    )